# Pelagic spin-up validation notebook

This notebook follows the following workflow:

1. **Configuration** for paths, variables, layers, and output folders.
2. **Helper functions** for loading model output and handling grid/time quirks.
3. **Analysis functions** for trends, subdomain diagnostics, observations, and satellite comparison.
4. **Entry-point examples** that can be toggled on when you want to run a specific workflow.

The main cleanup change is that map-based diagnostics now **export individual PNG frames** instead of rendering animations inline in the notebook.


In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
from IPython.display import display
import os


In [ ]:
# -----------------------------------------------------------------------------
# CONFIGURATION (minimal)
# -----------------------------------------------------------------------------

POSTPROC_DIR = Path('/export/lv9/projects/dws/results/validation/pelagic/')
BASE_OUTPUT_DIR = Path('/export/lv9/projects/dws/model_output/archived_runs/')

spinups = sorted([d for d in os.listdir(base_dir) if d.startswith("spinup_")])
#spinups = ["spinup_06"]
frame_counter = 0

DEFAULT_SPINUP = 'spinup_06'
YEAR = 2015
NC_PATTERN = 'dws_500m.3d.{year}??.nc'

MODEL_SURFACE_LAYER_INDEX = 10

# Choose ONE variable here
variable = "P1c"
label = "Diatoms (P1c)"

# variable = "P2c"
# label = "Flagellates (P2c)"

# variable = "P3c"
# label = "PicoPhytoPlankton (P3c)"

FRAME_OUTPUT_DIR = (
    POSTPROC_DIR / DEFAULT_SPINUP / f"{YEAR}_spinup_frames" / variable
)
FRAME_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Color scale — adjust if needed
VMIN = 0.0
VMAX = 200.0
CMAP = "viridis"

# Optional: mask out dry areas using elevation
MASK_VAR = "elev"
MASK_THRESHOLD = -1.1869



In [ ]:
# -----------------------------------------------------------------------------
# HELPER
# -----------------------------------------------------------------------------

def export_map_frames(
    ds: xr.Dataset,
    *,
    plotvar: str,
    output_dir: Path,
    layer_index: int | None,
    time_step: int,
    cmap: str,
    vmin: float,
    vmax: float,
    mask_var: str | None = None,
    mask_threshold: float | None = None,
) -> None:
    _apply_theme()
    if plotvar not in ds.variables:
        raise KeyError(f"Variable '{plotvar}' not found. Available: {sorted(ds.data_vars)}")
    if vmin is None or vmax is None:
        raise ValueError('vmin and vmax must both be provided for consistent frame export.')

    da = ds[plotvar].squeeze(drop=True)
    time_dim = _find_time_dim(da)
    field, z_dim = _resolve_layer(da, layer_index, time_dim)
    field = _drop_duplicate_time(field, time_dim)
    y_dim, x_dim = _find_xy_dims(field, (time_dim,))
    x_plot, y_plot, use_geo = _build_horizontal_coords(ds, y_dim, x_dim)
    layer_text = '' if z_dim is None else f' | layer {layer_index}'

    if mask_var is not None:
        if mask_var not in ds.variables:
            raise KeyError(f"Mask variable '{mask_var}' not found.")
        mask_da = ds[mask_var].squeeze(drop=True)
        mask_time_dim = _find_time_dim(mask_da)
        mask_da = _drop_duplicate_time(mask_da, mask_time_dim)
        field, mask_da = xr.align(field, mask_da, join='inner')
    else:
        mask_da = None

    frame_indices = np.arange(0, field.sizes[time_dim], time_step, dtype=int)
    if frame_indices.size == 0:
        raise ValueError('No frames selected. Check time_step and time length.')

    output_dir.mkdir(parents=True, exist_ok=True)
    print(f'Exporting {frame_indices.size} frames to: {output_dir}')

    for frame_number, frame_idx in enumerate(frame_indices):
        frame2d = _to_float(field.isel({time_dim: int(frame_idx)}).transpose(y_dim, x_dim).values)
        if mask_da is not None:
            wet_mask = _to_float(mask_da.isel({time_dim: int(frame_idx)}).transpose(y_dim, x_dim).values) > mask_threshold
            frame2d[~wet_mask] = np.nan

        timestamp = pd.Timestamp(field[time_dim].values[int(frame_idx)])
        frame_name = f"frame_{frame_number:04d}_{_format_timestamp_for_filename(timestamp)}.png"
        frame_path = output_dir / frame_name

        fig, ax = plt.subplots(figsize=(8, 6))
        mesh = ax.pcolormesh(
            x_plot,
            y_plot,
            np.ma.masked_invalid(frame2d),
            shading='auto',
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
        )
        cbar = fig.colorbar(mesh, ax=ax, fraction=0.035, pad=0.03)
        units = da.attrs.get('units', '')
        cbar.set_label(f'{plotvar} [{units}]' if units else plotvar)
        ax.set_title(f'{plotvar}{layer_text} | {timestamp}')
        ax.set_xlabel('Longitude' if use_geo else x_dim)
        ax.set_ylabel('Latitude' if use_geo else y_dim)
        ax.grid(True, alpha=0.2)
        plt.tight_layout()
        fig.savefig(frame_path, dpi=200, bbox_inches='tight')
        plt.close(fig)

    print(f'Saved {frame_indices.size} frames: {output_dir}')



In [ ]:

# -----------------------------------------------------------------------------
# FRAME EXPORT
# -----------------------------------------------------------------------------

def export_variable_frames(ds: xr.Dataset) -> None:
    print(f"Exporting frames for {variable} …")

    export_map_frames(
        ds,
        plotvar=variable,
        output_dir=FRAME_OUTPUT_DIR,
        layer_index=MODEL_SURFACE_LAYER_INDEX,
        time_step=1,
        cmap=CMAP,
        vmin=VMIN,
        vmax=VMAX,
        mask_var=MASK_VAR,
        mask_threshold=MASK_THRESHOLD,
    )

    print(f"Frames saved to: {FRAME_OUTPUT_DIR}")


# -----------------------------------------------------------------------------
# MAIN EXECUTION
# -----------------------------------------------------------------------------

print(f"Opening dataset for {DEFAULT_SPINUP} ({YEAR}) …")
ds = open_spinup_dataset(DEFAULT_SPINUP, YEAR)

try:
    export_variable_frames(ds)
finally:
    ds.close()
